In [1]:
import pandas as pd
from pathlib import Path

base = Path('/Users/estebanbigmac/Desktop/My huge Python/RA Sameer')

# 拷貝版:10/07 + 10/08 新增的部分
copy_dir = base / 'Anotherway' / 'Concat'
old_path = copy_dir / 'notes_2026-10-07.parquet'
if not old_path.exists():                          # Anotherway 沒有 10/07 時,改用原版的 10/07
    old_path = base / 'Concat' / 'notes_2026-10-07.parquet'
copy_df = pd.concat([pd.read_parquet(old_path),
                     pd.read_parquet(copy_dir / 'notes_2026-10-08.parquet')], ignore_index=True)

# 原版:10/08 本身就是累積的
orig_df = pd.read_parquet(base / 'Concat' / 'notes_2026-10-08.parquet')

# 欄位順序和型態對齊,不然內容相同的列雜湊值也會不同
copy_df = copy_df[orig_df.columns].astype(orig_df.dtypes)

copy_hash = pd.util.hash_pandas_object(copy_df, index=False)
orig_hash = pd.util.hash_pandas_object(orig_df, index=False)

only_copy = copy_df[(~copy_hash.isin(orig_hash)).values]
only_orig = orig_df[(~orig_hash.isin(copy_hash)).values]

print(f'10/07 來源:{old_path}')
print(f'拷貝版(合併後):{len(copy_df):,} 筆')
print(f'原版:{len(orig_df):,} 筆')
print(f'只在拷貝版:{len(only_copy):,} 筆')
print(f'只在原版:{len(only_orig):,} 筆')

10/07 來源:/Users/estebanbigmac/Desktop/My huge Python/RA Sameer/Anotherway/Concat/notes_2026-10-07.parquet
拷貝版(合併後):3,162,127 筆
原版:3,162,127 筆
只在拷貝版:0 筆
只在原版:0 筆


In [2]:
for label, df in [('只在原版', only_orig), ('只在拷貝版', only_copy)]:
    if len(df):
        t = pd.to_datetime(df['createdAtMillis'], unit='ms')
        print(f'{label}:{len(df):,} 筆,建立時間 {t.min()} ~ {t.max()}')
        print(f'  其中 noteId 已存在於另一邊:{df["noteId"].isin((only_copy if label == "只在原版" else orig_df)["noteId"]).sum():,} 筆')

In [3]:
orig_df

,noteId,noteAuthorParticipantId,createdAtMillis,tweetId,classification,believable,harmful,validationDifficulty,misleadingOther,misleadingFactualError,...,misleadingSatire,notMisleadingOther,notMisleadingFactuallyCorrect,notMisleadingOutdatedButNotWhenWritten,notMisleadingClearlySatire,notMisleadingPersonalOpinion,trustworthySources,summary,isMediaNote,isCollaborativeNote
0,1783179305159200982,C784F04F26E124F4D6EC01658D8F5565005D3092741FB3...,1713978050878,1783159712986382830,MISINFORMED_OR_POTENTIALLY_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,1,The House failed to pass a border protection l...,0,0
1,1783181538789605871,C784F04F26E124F4D6EC01658D8F5565005D3092741FB3...,1713978583415,1783171851818021181,MISINFORMED_OR_POTENTIALLY_MISLEADING,NaN,NaN,NaN,0,1,...,0,0,0,0,0,0,1,The United States has 50 States https://da...,0,0
2,1783182562279494134,C784F04F26E124F4D6EC01658D8F5565005D3092741FB3...,1713978827435,1783154445682979015,MISINFORMED_OR_POTENTIALLY_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,1,TikTok only mentions “ban” and chooses to igno...,0,0
3,1883711635770196070,C784F04F26E124F4D6EC01658D8F5565005D3092741FB3...,1737946826294,1883619411774345444,MISINFORMED_OR_POTENTIALLY_MISLEADING,NaN,NaN,NaN,1,0,...,0,0,0,0,0,0,1,This could be considered a threat https://...,0,0
4,1537142913737428992,5684B38EB58FD8BE75ABA37F0BE040EC70380B002ADF9D...,1655318404027,1377030478167937024,MISINFORMED_OR_POTENTIALLY_MISLEADING,BELIEVABLE_BY_MANY,CONSIDERABLE_HARM,EASY,0,1,...,0,0,0,0,0,0,1,Forbes has a good rundown of the investigation...,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3162122,2107237526951874681,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1791239629148,2107236388437459067,NOT_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,0,Note not needed. Humorous commentary on the Ro...,0,1
3162123,2107236492557111439,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1791239391301,2107236388437459067,NOT_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,0,"The Roadster event is set for October 15, 2026...",0,1
3162124,2107267291088146833,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1791246802474,2107239895651512365,NOT_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,0,The 16-year-old was the driver of the Range Ro...,0,1
3162125,2107318270504161347,F75B04680623CD9A452496D4AB69A20ED730CE93652C71...,1791259003202,2107289124910841992,NOT_MISLEADING,NaN,NaN,NaN,0,0,...,0,0,0,0,0,0,0,Note not needed. They were hired as head coach...,0,1


In [ ]:
rt_orig = pd.read_parquet(base / 'Concat' / 'ratings_2026-10-07.parquet')
rt_orig

,noteId,raterParticipantId,createdAtMillis,version,agree,disagree,helpful,notHelpful,helpfulnessLevel,helpfulOther,...,notHelpfulArgumentativeOrBiased,notHelpfulOffTopic,notHelpfulSpamHarassmentOrAbuse,notHelpfulIrrelevantSources,notHelpfulOpinionSpeculation,notHelpfulNoteNotNeeded,ratedOnTweetId,ratingSourceBucketed,suggestion,suggestionId
0,1352796878438424576,6BC15A6669481F9CEF45602263C22A860662F750EF683F...,1611860252442,1,1,0,1,0,NaN,0,...,0,0,0,0,0,0,-1,DEFAULT,NaN,-1
1,1352796878438424576,FEB28D48AB0ACC2E768DBD320BA0E393FE7C657D904407...,1611673955881,1,1,0,0,1,NaN,0,...,0,0,0,0,0,0,-1,DEFAULT,NaN,-1
2,1352796878438424576,A8CE8EFCAFC6F6228EEFC20E211071E6DCB40B97A050B6...,1611623203338,1,0,0,0,1,NaN,0,...,0,0,0,0,0,0,-1,DEFAULT,NaN,-1
3,1352796878438424576,457AA0AE4D242090FF340BB7E0154AC3DF30904BEEF5C5...,1611852744990,1,1,0,1,0,NaN,0,...,0,0,0,0,0,0,-1,DEFAULT,NaN,-1
4,1352796878438424576,00647715E8F0DD5D4EFFE5210FCD0DC741C3B921218124...,1611388222120,1,1,0,1,0,NaN,0,...,0,0,0,0,0,0,-1,DEFAULT,NaN,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236482819,2106955186052772254,2B264D216C9F504A5E854E5D4065EAA83D2BDC7D9D68A7...,1791172793185,2,0,0,0,0,HELPFUL,0,...,0,0,0,0,0,0,2106832552027623517,DEFAULT,NaN,-1
236482820,2106955508435612072,7F16863BE3FB87264952B6B64F80825EDA9544AAAC92BB...,1791172740367,2,0,0,0,0,HELPFUL,0,...,0,0,0,0,0,0,2106839882203308109,DEFAULT,NaN,-1
236482821,2106955573795139670,90694FA089597DB306F1153935ECE848E566CD4A7F46D6...,1791172370169,2,0,0,0,0,HELPFUL,0,...,0,0,0,0,0,0,2106739618477412855,DEFAULT,NaN,-1
236482822,2106956482054963347,21DE740A6FDA486DCD56C8747D67602592D2B54163562B...,1791172559883,2,0,0,0,0,HELPFUL,0,...,0,0,0,0,0,0,2106833162005528663,DEFAULT,NaN,-1


: 

In [ ]:
!pip3 install duckdb